# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramithnayak8/ML_pipeline/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

Same row selection as the reference pipeline (`impressions_90d > 0`, `content_age_days >= 90`,
deduped by `content_id`) so my numbers stay comparable -- on this already-curated starter slice
that filter turns out to be a no-op (confirmed below). I add two explicit missingness flags the
reference script skips: `has_keyword_data` and `has_word_count`. `w03_data_contract.ipynb`
confirmed missingness follows `content_type`, not chance, so a blind `fillna(0)` on
`search_volume` would silently tell the model "this is a feedly article" through the zero
itself. Flagging it first, then filling, keeps that information explicit instead of hidden.

In [1]:
import numpy as np
import pandas as pd

raw = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"raw rows: {len(raw):,}")

df = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
print(f"rows after the (no-op) filter + de-dup: {len(df):,}")

# missingness flags -- BEFORE filling, so they capture the real gap
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)

numeric_fill_zero = ["search_volume", "competition", "cpc", "word_count", "char_count",
                     "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
                     "days_with_impressions", "days_with_sessions", "content_age_days",
                     "days_since_last_update", "ctr", "avg_position", "engagement_rate",
                     "scroll_rate", "ai_traffic_pct"]
for col in numeric_fill_zero:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)

categorical_cols = ["competition_level", "content_type", "main_intent", "age_tier",
                    "freshness_tier", "word_count_tier", "impression_tier", "position_tier"]
for col in categorical_cols:
    df[col] = df[col].fillna("unknown").astype(str)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

feature_cols = (["search_volume", "competition", "cpc", "word_count", "char_count",
                 "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
                 "days_with_impressions", "days_with_sessions", "content_age_days",
                 "days_since_last_update", "ctr", "avg_position", "engagement_rate",
                 "scroll_rate", "ai_traffic_pct", "has_keyword_data", "has_word_count"]
                + categorical_cols)

print(f"\n{len(feature_cols)} raw feature columns before one-hot encoding")
df[feature_cols + ["is_declining_label"]].head(3)


raw rows: 30,000
rows after the (no-op) filter + de-dup: 30,000

28 raw feature columns before one-hot encoding


,search_volume,competition,cpc,word_count,char_count,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,days_with_impressions,...,has_word_count,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier,is_declining_label
0,10.0,0.67,2.05,3221.0,20457.0,8.243808,3.401197,2.890372,0.0,88,...,1,HIGH,keyword article,transactional,181-365,0-30,2000-3500,good,striking,1
1,90.0,0.01,0.05,2481.0,15562.0,9.636980,2.079442,2.302585,0.0,88,...,1,LOW,keyword article,informational,365+,0-30,2000-3500,good,page_3_5,1
2,0.0,0.00,0.00,3515.0,23643.0,9.440023,2.484907,2.484907,0.0,88,...,1,LOW,keyword article,informational,91-180,0-30,3500+,good,page_3_5,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

- `search_volume`, `competition`, `cpc` -- keyword-tool estimates; blank for pages with no
  matched keyword (concentrated in `feedly article`, confirmed in `w03_data_contract.ipynb`);
  filled with 0 plus the `has_keyword_data` flag. Available before prediction -- content
  metadata, not an outcome measurement.
- `word_count`, `char_count` -- article size; blank for ~28% of `keyword article` rows; filled
  with 0 plus `has_word_count`. Available before prediction -- a static content fact.
- `avg_position`, `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct` -- derived rates over
  the trailing 90-day window (NOT the label's 30-day sub-windows). `avg_position == 0` is the
  documented "no data" code, left as 0 deliberately since it already IS the correct signal for
  "no GSC position recorded." Available before prediction -- they summarize the past 90 days,
  not the outcome.
- `days_since_last_update`, `content_age_days` -- static content facts, never missing, always
  available before prediction.
- `has_keyword_data`, `has_word_count` -- engineered flags; no missingness by construction, since
  they are defined FROM the missingness itself.
- 8 categorical fields (`content_type`, tiers, etc.) -- filled with the string `"unknown"` when
  blank, one-hot encoded downstream.

In [2]:
print("NaNs remaining in feature columns after fill:", df[feature_cols].isna().sum().sum())
print("has_keyword_data == 0 (no keyword data):", (df["has_keyword_data"] == 0).sum(), "rows")
print("has_word_count == 0 (no word count):", (df["has_word_count"] == 0).sum(), "rows")
print("avg_position == 0 (no position data):", (df["avg_position"] == 0).sum(), "rows")


NaNs remaining in feature columns after fill: 0
has_keyword_data == 0 (no keyword data): 2468 rows
has_word_count == 0 (no word count): 7699 rows
avg_position == 0 (no position data): 1205 rows


## 3. The leakage hunt

Attack test from the leakage-hunting checklist: train the SAME model once without the suspect
column, once with it, and watch for a collapse toward a too-good score. `trend_pct` is the
textbook case -- `trend_direction` (the label) is a threshold applied directly to it, so it is
the label's own ingredient, not an ordinary feature. If adding it barely moves the score, my test
harness itself would be broken; if it jumps the score toward a suspiciously perfect number,
that is the confession, not a win. Scored in-sample on purpose (this is an attack test on the feature set, not a validation claim -- honest out-of-fold validation is w06_validation_audit.ipynb's job).


In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = df["is_declining_label"]
print(f"base rate (always report this next to any score): {y.mean():.1%}")

X_clean = pd.get_dummies(df[feature_cols], drop_first=True)
X_leaky = X_clean.copy()
X_leaky["trend_pct"] = df["trend_pct"].fillna(0)

model_clean = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(X_clean, y)
model_leaky = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(X_leaky, y)

proba_clean = model_clean.predict_proba(X_clean)[:, 1]
proba_leaky = model_leaky.predict_proba(X_leaky)[:, 1]

print(f"\nWITHOUT trend_pct -- ROC-AUC: {roc_auc_score(y, proba_clean):.3f}   "
      f"P@50: {precision_at_k(proba_clean, y, 50):.3f}")
print(f"WITH    trend_pct -- ROC-AUC: {roc_auc_score(y, proba_leaky):.3f}   "
      f"P@50: {precision_at_k(proba_leaky, y, 50):.3f}   <- the confession")


base rate (always report this next to any score): 54.2%



WITHOUT trend_pct -- ROC-AUC: 0.709   P@50: 0.940
WITH    trend_pct -- ROC-AUC: 1.000   P@50: 1.000   <- the confession


## 4. What I excluded and why

Full detail lives in `w03_data_contract.ipynb`; the short version, verified against the actual
feature matrix below:
- `trend_direction`, `trend_pct` -- the label and its raw ingredient (just demonstrated above).
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`,
  `clicks_prev_30d`, `sessions_prev_30d` -- the exact arithmetic `trend_pct` is built from.
- `provider_used`, `model_used` -- content-generation metadata, not a performance signal.
- `content_id`, `client_id` -- pseudonyms; context/grouping only.

In [4]:
excluded = {"trend_direction", "trend_pct", "provider_used", "model_used",
            "content_id", "client_id", "impressions_last_30d", "clicks_last_30d",
            "sessions_last_30d", "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"}

present = excluded & set(X_clean.columns)
print("Excluded columns present in the final feature matrix?", present if present else "NONE (good)")
print("Final X_clean shape:", X_clean.shape)


Excluded columns present in the final feature matrix? NONE (good)
Final X_clean shape: (30000, 46)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.